# Week 4, day 2 (afternoon) — dbt on Snowflake: from raw to data marts

The WeCloudData **Create a dbt Project** lab and the **dbt Fundamentals**
lecture, as one notebook that runs **entirely inside Snowflake**.

You build a complete, tested dbt project — sources, staging, snapshots, a
**Type 6** slowly-changing dimension, a **star schema** on surrogate keys,
business-driven **seeds**, Great-Expectations-style quality checks, **unit
tests**, a **MetricFlow semantic layer**, and **lineage** across every layer —
then hand it to Snowflake to execute as a native `DBT PROJECT` object.

## How it works

| Cell type | Does |
|---|---|
| **SQL** | the warehouse work: schemas, raw tables, loading, `EXECUTE DBT PROJECT`, inspecting results, scheduling |
| **Python** | writes the dbt project files, and uploads them to a stage |

**There are no credentials in this notebook.** `get_active_session()` gives you
the session you are already authenticated in, and dbt runs as the executing role.
Nothing to configure, nothing to leak.

## Before you start

- A role that can create a database, schemas, stages, tasks and a `DBT PROJECT`,
  and a warehouse attached to this notebook.
- The two CSVs to hand: `products.csv` (1,214 rows) and `sales.csv` (100,000
  rows), from the lab's `create_dbt_project_datasets.zip`. Question 5 uploads
  them through the Snowsight stage UI.

## The layers you are building

```
RAW  ──▶  STG  ──▶  EDW  ──▶  MARTS
 │         │         │          │
 │         │         │          ├─ rpt_*   hand-written SQL models
 │         │         │          └─ mart_*  exported from MetricFlow saved queries
 │         │         │
 │         │         └─ dim_product_t6 (Type 6) / dim_store / dim_date / fct_sales
 │         └─ stg_product_incr / stg_sales (views) + the snapshots
 └─ product / sales  (the two CSVs)
                    + seeds: store_master, category_targets
```

New to dbt? The day's README has a **concepts primer** — model, source, ref,
seed, snapshot, materialization, macro, test — read that first.

Work the questions in order: each builds on the objects the earlier ones made.


## A dbt primer — read this first

Everything below assumes this vocabulary. If you already know dbt, skip to Part A.

### What dbt is

Modern pipelines are **ELT**: data is *loaded* into the warehouse first, then
*transformed* inside it. dbt owns that **T**. It moves no data and has no runtime
of its own — it compiles your SQL and asks Snowflake to run it.

You write `SELECT` statements. dbt works out the order, wraps each in the right
`CREATE TABLE` / `CREATE VIEW` / `MERGE`, runs your tests, and builds the
documentation and lineage graph.

What it replaces: a folder of numbered `.sql` scripts where logic is
copy-pasted, dependencies live in someone's head, and "is this number right?" has
no answer.

### The objects

| Concept | What it is | Here |
|---|---|---|
| **Model** | a `.sql` file holding one `SELECT`; dbt materializes it | `stg_sales`, `dim_product_t6`, `fct_sales` |
| **Source** | a raw table dbt did *not* build, declared so it can be referenced and tested | `raw.product`, `raw.sales` |
| **`ref()`** | how one model reads another — this is what builds the DAG | everywhere |
| **`source()`** | how a model reads a raw table | staging models |
| **Seed** | a CSV in the project, loaded by `dbt seed` | `store_master`, `category_targets` |
| **Snapshot** | records how a *mutable* source changes over time | `product_snapshot` |
| **Test** | an assertion about the data; failures stop the build | `unique`, `not_null`, `relationships` |
| **Unit test** | an assertion about a model's *logic*, against mock rows | the Type 6 scenario |
| **Macro** | a Jinja function returning SQL | `to_active_flag` |
| **Package** | an installable library of macros and tests | `dbt_utils`, `dbt_expectations` |
| **Metric** | a business definition written once, in the semantic layer | `margin_rate` |

**Never hardcode a table name — always `ref()` or `source()`.** That one habit is
what gives dbt the dependency graph; build order, lineage and impact analysis all
follow from it.

### Materializations

| Materialization | dbt does | Use when |
|---|---|---|
| **view** (default) | `CREATE VIEW` each run | light transformation, no storage, always fresh |
| **table** | `CREATE TABLE` each run | queried often, rebuild is affordable |
| **incremental** | insert/update only new rows | full rebuild too slow |
| **ephemeral** | nothing — inlined as a CTE | a small step one or two models need |

**Rule of thumb:** start with a view; when it is slow to query make it a table;
when the table is slow to build make it incremental.

### Configuring a model — three routes, one precedence order

A model's settings (`materialized`, `tags`, `schema`, `persist_docs`, incremental
options…) can be declared in three places. **Highest priority first:**

| # | Where | Scope | Use for |
|---|---|---|---|
| 1 | a `config()` block **inside the `.sql`** | that model | things intrinsic to the query — an incremental strategy, a `unique_key` |
| 2 | a `config:` block in a **properties `.yml`** | that model | per-model settings you want beside the description and tests |
| 3 | **`dbt_project.yml`** under `models:` | a whole folder | defaults for a layer |

A project is therefore built largely **in YAML**: `dbt_project.yml` defines the
project and the folder defaults, and the properties files carry per-model
configuration, documentation and tests. `sources`, `seeds`, `snapshots`,
`exposures`, `unit_tests`, semantic models and metrics are *only* definable in
YAML — the `.sql` files hold the `SELECT` and nothing else.

Elsewhere you would scaffold all this with **`dbt init <name>`**, which creates
the folder tree and interviews you for the connection. There is no shell in a
Snowflake notebook, so this lab writes the files directly — same result.

### Incremental models

Three parts, always: a `config` marking it incremental, an `is_incremental()`
block holding a **cutoff filter**, and the filter itself (usually against
`this`, the model's existing table). First run builds everything; later runs
process only rows past the watermark.

Two strategies appear here:

- **`merge`** — update matching rows, insert the rest. For keyed staging rows
  (`stg_product_incr`).
- **`delete+insert`** — delete the matching keys, then insert. For recomputed
  **aggregates**, which must be *replaced* rather than merged (`fct_sales`).

### Slowly-changing dimensions, and why Type 6

A source row changes and the old value is lost — unless you capture it.

| Type | Behaviour |
|---|---|
| **Type 1** | overwrite; only "now" exists |
| **Type 2** | a new row per version, with validity dates |
| **Type 3** | keep a `previous_*` column |
| **Type 6** | **1 + 2 + 3 together** (1 × 2 × 3 = 6) |

`dim_product_t6` carries all three on every row:

| View | Column | Answers |
|---|---|---|
| Type 2 | `category_name` | what was it **at the time**? |
| Type 1 | `current_category_name` | re-state history under **today's** value |
| Type 3 | `previous_category_name` | what did it **change from**? |

So the same fact row can be reported "as was" or "as is" without touching the
fact table. The **snapshot** supplies the Type 2 raw material
(`dbt_valid_from` / `dbt_valid_to`); the model derives the other two.

### Star schema and surrogate keys

A **fact** surrounded by **dimensions**, joined on **surrogate keys**.

```
                  dim_date
                      │
   dim_product_t6 — fct_sales — dim_store
```

The fact holds only keys and **measures**, at a stated **grain** (one row per
date / product version / store) — every measure must be additive at that grain.
Descriptions live in the dimensions, so a category rename never touches the fact.

A **surrogate key** is a warehouse-generated id with no business meaning
(`dbt_utils.generate_surrogate_key` hashes `prod_key` + `valid_from`). Why not
just `prod_key`? In an SCD it is **not unique** — a product has one row per
version. The surrogate key is unique per *version*, which is what lets the fact
point at exactly the right one.

### The layers

| Layer | Schema | Contains | Materialized |
|---|---|---|---|
| **raw** | `RAW` | the two loaded CSVs; dbt only reads these | — |
| **staging** | `STG` | 1:1 with sources: rename, cast, select. No joins or aggregation | views |
| **edw** | `EDW` | conformed dimensions and facts — the star schema | tables |
| **marts** | `MARTS` | business-facing reports | tables |

Each layer is a **folder** *and* a **schema** — which is what makes the lineage
readable and lets you build one layer at a time.

### Tests

- A **data test** runs against the warehouse; it compiles to a query returning
  offending rows, so zero rows = pass. The four generic ones are `unique`,
  `not_null`, `accepted_values`, `relationships`.
- **dbt-expectations** (the dbt port of Great Expectations) adds range,
  distribution, shape and freshness checks — the defects that pass every
  column-level test and still make a report wrong.
- A **unit test** feeds a model mock rows and asserts exact output. It checks
  *logic*, with no warehouse data — ideal for SCD rules.

### The commands

Inside Snowflake every one of these runs as
`EXECUTE DBT PROJECT <name> ARGS = '<command>'`:

```
deps            install packages
seed            load the CSV seeds
run             build models only
snapshot        build snapshots only (stateful — advances history)
test            run tests only
build           seeds + snapshots + models + tests, in DAG order   <- schedule this
ls --select +x  what feeds x?      ls --select x+   what breaks if x changes?
build --full-refresh               rebuild incrementals from scratch
```

`build` stops a downstream model when an upstream test fails, so bad data does
not propagate. That is why it is the command you schedule.


Run this cell first, every session. It picks up the session you are already
authenticated in — no account, user or password anywhere — and defines the names
every later cell uses. If `CURRENT_WAREHOUSE()` comes back null, attach a
warehouse to the notebook before going on.

In [ ]:
from snowflake.snowpark.context import get_active_session
from pathlib import Path

session = get_active_session()

# Everything the project needs, in one place. Change these if your account uses
# different names; every cell below reads them.
DB    = "DEMO_DB"
WH    = session.get_current_warehouse().strip('"')
ROLE  = session.get_current_role().strip('"')
PROJ  = "/tmp/demo"          # where the dbt project is assembled in this notebook
STAGE = f"{DB}.RAW.DBT_PROJECT_STAGE"

session.sql("SELECT CURRENT_ROLE(), CURRENT_WAREHOUSE(), CURRENT_VERSION()").show()
print("database :", DB)
print("warehouse:", WH)
print("role     :", ROLE)

## PART A — the environment

A Snowflake notebook already *is* connected: `get_active_session()` hands you the
session for the role, warehouse and database you are running as. There is no
`profiles.yml`, no account locator, no password — anywhere in this notebook.

That is the single biggest difference from running dbt on a laptop, and it is why
this whole lab fits in one notebook.

### Question 1

Create the **warehouse** this lab runs on, and check what privileges your
role has. Skip the `CREATE WAREHOUSE` if you already have one attached.

In [ ]:
-- Your Code Here

### Question 2

**Optional, ACCOUNTADMIN only.** Create a dedicated role for the lab and grant
it what it needs. Skip this if you are already running as `SYSADMIN` or
similar — the rest of the notebook does not depend on it.

In [ ]:
-- Your Code Here

### Question 3

Create the database and the five schemas, one per layer.

In [ ]:
-- Your Code Here

### Question 4

Create the two raw tables, matching the CSV headers exactly.

In [ ]:
-- Your Code Here

### Question 5

Create a named **file format** for the CSVs, so the parsing rules are one
object every `COPY` reuses instead of being retyped.

In [ ]:
-- Your Code Here

### Question 6

Create the stages: one for the CSVs, one to hold the dbt project.

In [ ]:
-- Your Code Here

### Question 7

Upload `products.csv` and `sales.csv` to `RAW.LOAD_STAGE`, then load them.

Upload first — in Snowsight: **Data → Databases → DEMO_DB → RAW → Stages →
LOAD_STAGE → + Files**. Then run the `COPY INTO`.

In [ ]:
-- Your Code Here

### Question 8

Verify the load: row counts, and the date range that proves the date format
was applied.

In [ ]:
-- Your Code Here

## PART B — writing the dbt project

Now the transformation layer. You assemble a complete dbt project as files, then
hand it to Snowflake to run.

The cells write to `/tmp/demo` inside the notebook, using plain `pathlib` — no
shell and no magics, both of which a Snowflake notebook may not give you.

### Question 9

Create the project skeleton — one folder per layer, plus macros, seeds,
snapshots and analyses.

On a laptop you would not do this by hand: **`dbt init demo`** scaffolds exactly
this tree and then interviews you for the connection details. There is no shell
in a Snowflake notebook, so we create the folders directly — but `dbt init` is
the command you will use everywhere else.

In [ ]:
############################
## Your Code Here
############################

### Question 10

Write `dbt_project.yml` — the file that makes a directory a dbt project. It
routes each folder to its schema and sets the project vars.

In [ ]:
############################
## Your Code Here
############################

### Question 11

Write `profiles.yml`. Running inside Snowflake, it carries **no credentials** —
the executing role is the identity.

In [ ]:
############################
## Your Code Here
############################

### Question 12

Write `packages.yml` — dbt_utils (surrogate keys) and dbt-expectations (the
Great-Expectations-style tests).

In [ ]:
############################
## Your Code Here
############################

### Question 13

Write the custom `generate_schema_name` macro so the layer schemas are used
verbatim.

In [ ]:
############################
## Your Code Here
############################

### Question 14

Declare the sources — the raw tables you loaded in Part A.

In [ ]:
############################
## Your Code Here
############################

### Question 15

Write the two staging models: `stg_product_incr` (incremental, `merge`
strategy, stamping a `start_date`) and `stg_sales`.

In [ ]:
############################
## Your Code Here
############################

### Question 16

Write `stg_sales` — rename and select only, no joins, no aggregation.

In [ ]:
############################
## Your Code Here
############################

### Question 17

Document the staging layer — a reusable doc block, then the descriptions and
tests that consume it.

In [ ]:
############################
## Your Code Here
############################

### Question 18

Write the staging model properties.

In [ ]:
############################
## Your Code Here
############################

## PART C — seeds, snapshots, and the Type 6 dimension

**Seeds** are reference data in the repo. **Snapshots** capture how a mutable
source changes. Together they feed a **Type 6** slowly-changing dimension and the
star schema around it.

### Question 19

The business reports by region, but sales only has a `store_key`. Write the
`store_master` seed to bridge that gap.

In [ ]:
############################
## Your Code Here
############################

### Question 20

Finance measures actuals against plan, and the targets exist only in a
spreadsheet. Write the `category_targets` seed, and the seed properties.

In [ ]:
############################
## Your Code Here
############################

### Question 21

Write `seeds.yml` — descriptions and tests for both seeds.

In [ ]:
############################
## Your Code Here
############################

### Question 22

Write the product snapshot, using the **check** strategy.

In [ ]:
############################
## Your Code Here
############################

### Question 23

Write the sales snapshot using the **timestamp** strategy, in the modern YAML
form.

In [ ]:
############################
## Your Code Here
############################

### Question 24

Write the `to_active_flag` macro — the "null close date means current" rule,
written once.

In [ ]:
############################
## Your Code Here
############################

### Question 25

Write `dim_product_t6` — the **Type 6** dimension, keyed by a surrogate key.

In [ ]:
############################
## Your Code Here
############################

### Question 26

Write `dim_store` (from the seed) and `dim_date`.

In [ ]:
############################
## Your Code Here
############################

### Question 27

Write `dim_date`, derived from the dates present in sales.

In [ ]:
############################
## Your Code Here
############################

### Question 28

Write `fct_sales` — the fact at the centre of the star, incremental with the
**delete+insert** strategy.

In [ ]:
############################
## Your Code Here
############################

### Question 29

Write the EDW properties: documentation plus all four generic tests, including
referential integrity from fact to each dimension.

In [ ]:
############################
## Your Code Here
############################

### Question 30

Write the **unit tests** — automated scenarios for the Type 6 logic.

In [ ]:
############################
## Your Code Here
############################

## PART D — marts, quality, and the semantic layer

### Question 31

Write the pivot macro — it generates one SUM column per category with a Jinja
loop.

In [ ]:
############################
## Your Code Here
############################

### Question 32

Write the three mart models: sales by region, actuals vs target, and the
macro-generated pivot.

In [ ]:
############################
## Your Code Here
############################

### Question 33

Write `rpt_category_vs_target` — the report the seed exists for.

In [ ]:
############################
## Your Code Here
############################

### Question 34

Write `rpt_category_pivot`, calling your macro.

In [ ]:
############################
## Your Code Here
############################

### Question 35

Write two **custom generic tests** of your own, in `tests/generic/` —
a row-count check and a range check. These replace what dbt-expectations would
have given us, without the package.

In [ ]:
############################
## Your Code Here
############################

### Question 36

Write the second one — a column range check.

In [ ]:
############################
## Your Code Here
############################

### Question 37

Write the marts properties file. Note this one also **configures** the
models in YAML — `materialized`, `tags`, `persist_docs` — rather than with a
`config()` block in the `.sql`.

In [ ]:
############################
## Your Code Here
############################

### Question 38

Prove the **precedence** rule. A model can be configured three ways —
show where each one lives for `rpt_sales_by_region`, and which wins.

In [ ]:
############################
## Your Code Here
############################

### Question 39

Write the MetricFlow **time spine** — a dense, gap-free calendar.

In [ ]:
############################
## Your Code Here
############################

### Question 40

Write the semantic models — entities, dimensions and measures over the star
schema.

In [ ]:
############################
## Your Code Here
############################

### Question 41

Write the metrics — simple, ratio, derived, cumulative and filtered.

In [ ]:
############################
## Your Code Here
############################

### Question 42

Write the saved queries whose exports become **extra data marts**.

In [ ]:
############################
## Your Code Here
############################

### Question 43

Write the exposure and the analysis, then list every file you have created.

In [ ]:
############################
## Your Code Here
############################

### Question 44

Write the analysis, then inventory the whole project.

In [ ]:
############################
## Your Code Here
############################

## PART E — hand the project to Snowflake

The files exist in the notebook. Now they become a **native Snowflake object**:
upload to a stage, create a `DBT PROJECT`, and execute dbt commands as SQL.

> **The packages problem.** `dbt deps` fetches dbt_utils and dbt_expectations from
> the internet, and code inside Snowflake has no outbound network by default.
> Either vendor `dbt_packages/` into the upload, or have an ACCOUNTADMIN create a
> network rule + external access integration for `hub.getdbt.com`. Question 41
> shows both.

### Question 45

Upload every project file to the stage, preserving the folder structure.

In [ ]:
############################
## Your Code Here
############################

### Question 46

Create the dbt project object from the stage.

In [ ]:
-- Your Code Here

### Question 47

Build everything: seeds, snapshots, models and tests, in dependency order.

In [ ]:
-- Your Code Here

### Question 48

Build one layer at a time, and run a single model — the same selectors as the
CLI.

In [ ]:
-- Your Code Here

### Question 49

Materialize the semantic layer's saved queries — the extra marts.

In [ ]:
-- Your Code Here

## PART F — lineage across the layers

dbt knows the whole graph because every model declares its inputs with `ref()`
and `source()`. Both directions are answerable — and the second one is the
question to ask *before* editing anything.

### Question 50

Trace **upstream**: everything that feeds the regional sales report.

In [ ]:
-- Your Code Here

### Question 51

Trace **downstream**: the blast radius if `stg_sales` changed.

In [ ]:
-- Your Code Here

### Question 52

Confirm the layers landed where the routing said they would.

In [ ]:
-- Your Code Here

## PART G — inspect what you built

### Question 53

Look at the Type 6 dimension: all three views on one row.

In [ ]:
-- Your Code Here

### Question 54

Query the star: the fact joined to all three dimensions.

In [ ]:
-- Your Code Here

### Question 55

Compare the hand-written mart with the semantic-layer export.

In [ ]:
-- Your Code Here

### Question 56

Make history actually happen: change a product, re-snapshot, rebuild, and watch
the Type 6 columns diverge.

In [ ]:
-- Your Code Here

## PART H — scheduling

A project is only useful if it runs without you. Inside Snowflake the scheduler
is already there — a `TASK`, no cron and no external orchestrator.

### Question 57

Create a task that builds the project every morning, and start it.

In [ ]:
-- Your Code Here

### Question 58

Check the task history — how you find out a scheduled run failed.

In [ ]:
-- Your Code Here

## PART I — what dbt actually created, and cleaning up

dbt wrote SQL on your behalf. Snowflake will show you exactly what, which is the
best way to close the loop: you wrote a `SELECT`, and *this* is the object that
came out.

### Question 59

Ask Snowflake for the DDL of the objects dbt built. Compare a view, a table
and the fact.

In [ ]:
-- Your Code Here

### Question 60

Show every object the lab created, grouped by layer, with row counts.

In [ ]:
-- Your Code Here

### Question 61

**Tear it down.** Everything the lab created, in one cell — so you can re-run
the class from scratch, and so nothing keeps billing.

In [ ]:
-- Your Code Here